In [ ]:
# --timeframe 1d   : Таймфрейм свечей (дневные данные).
# --start-year 2000: Глубина загрузки истории (начиная с 2000 года).
# --workers 6      : Количество параллельных потоков для ускорения загрузки.

!python -m _tools.update_market_data --timeframe 1d --start-year 2000 --workers 6
!python -m _tools.update_macro --timeframe 1d --start-year 2000 --workers 6
# Выполняет комплексную проверку целостности, отсутствия пропусков и корректности OHLCV данных.
!python -m _tools.check_data_quality

In [ ]:
# --timeframe 1d       : Интервал данных — дневные свечи.
# --lookback 60        : Глубина истории — модель смотрит на 60 дней назад.
# --horizon 10         : Горизонт прогноза — ищем выход по барьерам в течение 10 дней.
# --auto               : Режим автоматического расчета уровней TP/SL на основе волатильности.
# --percentile 75      : Перцентиль волатильности для отсечения аномальных выбросов при авто-разметке.
# --init_split         : Дата начала первого разделения данных на Train и Val.
# --val_interval 2     : Продолжительность валидационного периода в годах.
# --split_interval 2   : Шаг смещения окна Walk-Forward в годах.
# --endpoint           : Дата окончания формирования всех временных интервалов.
# --corr_threshold     : Порог удаления коррелирующих признаков (убираем дубликаты > 85%).
# --cum_threshold      : Порог кумулятивной важности (оставляем топ фичей, дающих 99% влияния).
# --force              : Раскомментируйте параметр ниже для полной перезаписи кэшированных данных.

!python -m _tools.init_dataset \
    --timeframe 1d \
    --lookback 60 \
    --horizon 10 \
    --auto \
    --percentile 75 \
    --init_split 2010-01-01 \
    --val_interval 2 \
    --split_interval 2 \
    --endpoint 2024-01-01 \
    --corr_threshold 0.85 \
    --cum_threshold 0.99 \
    #--force

In [24]:
!python -m _tools.verify_data

I0000 00:00:1776764278.333671  261702 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776764279.941755  261702 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
✅ Расширенный аудит завершен: /home/restorator/trader_test/data_audit_report.txt


In [ ]:
#!rm -rf data/processed

In [ ]:
import os
import subprocess
from pathlib import Path

DATASET_DIR = Path("data/processed/2000_2026_1d")
print("🚀 Запуск массового обучения моделей (Walk-Forward)...")

# Получаем список папок fold_
folds = sorted([d for d in DATASET_DIR.glob("fold_*") if d.is_dir()])

try:
    for fold_dir in folds:
        fold_name = fold_dir.name
        
        # --- ПРОВЕРКА УДАЛЕНА ---
        # Мы убрали 'if list(models_dir.glob("*.keras")): continue'
        # Теперь внешний скрипт всегда вызывает train_model, 
        # а тот уже сам читает флаг --append и добавляет модели!
        
        print("="*60)
        print(f"🔥 Обучение (добавление) нейросети для: {fold_name}")
        print("="*60)
        
        # Формируем команду вызова
        cmd = [
            "python", "-m", "_tools.train_model",
            "--dataset_dir", str(DATASET_DIR),
            "--fold", fold_name,
            "--runs", "50",
            "--batch_size", "8192", 
            "--epochs", "50",
            "--l2_reg", "1e-4",
            "--lr", "1e-3",
            "--append"  # <--- Теперь этот флаг дойдет до адресата!
        ]
        
        # Запускаем процесс и позволяем ему выводить логи в реальном времени
        process = subprocess.Popen(cmd)
        
        # Ждем завершения, но позволяем Jupyter перехватить прерывание
        process.wait()
        
        print(f"✅ [{fold_name}] Завершен!")

except KeyboardInterrupt:
    print("\n🛑 Остановка пайплайна пользователем!")
    if 'process' in locals():
        process.terminate() # Мягкая остановка текущего процесса
        print("⏳ Завершаем текущий фолд...")
except Exception as e:
    print(f"❌ Ошибка: {e}")

print("🎉 Процесс полностью остановлен.")

In [ ]:
!python -m _tools.prepare_rl_env

In [ ]:
!python -m _tools.train_rllib_pbt --population 4 --iterations 3000

In [35]:
!python -m _tools.tournament_2026

Собираем топ претендентов...
🔍 Чтение статистики из training_summary.txt и поиск чекпоинтов...
                        ДОСТУПНЫЕ ЧЕКПОИНТЫ ДЛЯ ТУРНИРА                         
Trial ID        | Iter  | Train %    | Test %     | Status                   
--------------------------------------------------------------------------------
21cae_00003     | 457   | 43.76      | 23.17      | ✅ ПЕРСПЕКТИВНЫЙ (Robust) 
21cae_00002     | 498   | 28.31      | 12.58      | ✅ ПЕРСПЕКТИВНЫЙ (Robust) 
21cae_00000     | 538   | 14.92      | 9.36       | ✅ ПЕРСПЕКТИВНЫЙ (Robust) 

🏆 Запускаем турнир 2025-2026 годов для 3 моделей...

[1/3] Загрузка нейросети 21cae_00003_v457...
2026-04-22 07:01:58,066	WARNING rl_module.py:463 -- DeprecationWarning: `RLModule(config=[RLModuleConfig object])` has been deprecated. Use `RLModule(observation_space=.., action_space=.., inference_only=.., model_config=.., catalog_class=..)` instead. This will raise an error in the future!
   ⚠️ Ошибка: В среде нет данных за 202

In [36]:
!python -m _tools.test_single_2026


🚀 ЗАПУСК ИНДИВИДУАЛЬНОГО ТЕСТА ДЛЯ 2026 ГОДА
Модель: 7c64f_00003_v51
❌ Папка триала 7c64f_00003 не найдена!


In [34]:
!python _tools/check_ray_logs.py

🔍 Проверка директории: /home/restorator/trader_test/data/processed/2000_2026_1d/rl_env/ray_results/pbt_trading_bot
✅ Найдено 4 папок триалов.

📂 Анализ папки: PPO_TradingEnv-v0_21cae_00003_3_2026-04-21_23-53-21
   Найдено чекпоинтов: 458
   Примеры чекпоинтов: ['checkpoint_000219', 'checkpoint_000246', 'checkpoint_000280']
   ❌ Файл result.json не найден!
